# Fase 6 — Polars · CIC-IoT-2023

600.000 filas, 40 columnas · 10 consultas idénticas por motor. Los resultados se guardan en `06_fase6_procesamiento/results/<motor>/`. No se modifica la muestra original. Ejecutar **Run All** de arriba abajo.

## Configuración y carga

La carga queda fuera de los tiempos Q1–Q10. Las operaciones perezosas se materializan dentro de cada consulta.

In [1]:
import sys
from pathlib import Path
actual = Path.cwd().resolve()
for candidato in [actual, *actual.parents]:
    fase = candidato / "06_fase6_procesamiento"
    if (fase / "comun.py").is_file():
        sys.path.insert(0, str(fase))
        break
else:
    raise RuntimeError("Ejecutar el notebook dentro del repositorio")
import os
import platform
import polars as pl
from comun import (ENTRADA, ETIQUETAS, PROTOCOLOS, FASE, medir, cerrar, filas_tabla)

MOTOR = "polars"
TIEMPOS = []
df = pl.read_csv(ENTRADA)
TOTAL = df.height
assert TOTAL == 600_000 and len(df.columns) == 40
print("Motor:", MOTOR, pl.__version__, "| entorno:", platform.platform(), "| CPU:", os.cpu_count())
print("Entrada:", ENTRADA, "| registros:", TOTAL)


def familia(df_):
    return df_.with_columns(pl.col("Label").replace_strict(ETIQUETAS).alias("Attack_Family"))


def tipo(df_):
    return df_.with_columns(pl.when(pl.col("Label") == "Benign")
                            .then(pl.lit("Benign")).otherwise(pl.lit("Malicious"))
                            .alias("Traffic_Type"))


def limpio_rate(df_):
    # NULL para los 13 +inf; no se eliminan filas ni se limpia IAT.
    return df_.with_columns(pl.when(pl.col("Rate").is_finite())
                            .then(pl.col("Rate")).otherwise(None).alias("Rate"))




Motor: polars 1.44.2 | entorno: Windows-11-10.0.26200-SP0 | CPU: 16
Entrada: C:\Users\steve\Desktop\Big_Data\ProyectoBigData_CICIoT2023\01_fase1_datos\muestra\CICIoT2023_sample_600k.csv | registros: 600000


## Q1 — Valida las 600.000 filas originales (sin limpiar) y cuenta nulos, infinitos y duplicados exactos.

In [2]:
def q01():
    nulos = sum(df[c].null_count() for c in df.columns)
    infs = sum(int(df[c].is_infinite().sum()) for c, t in df.schema.items() if t.is_float())
    return [{"registros": TOTAL, "columnas": len(df.columns), "nulos": nulos,
             "infinitos": infs, "duplicados": TOTAL - df.unique().height}]


r01 = medir("Q1", q01, TIEMPOS, MOTOR)



Q1: 1 filas → q01_validacion.csv


  {'registros': 600000, 'columnas': 40, 'nulos': 18, 'infinitos': 13, 'duplicados': 131030}


Q1: 0.430 s (incluye cómputo/materialización, no exportación)



## Q2 — Deduplicación lógica sobre las 40 columnas; la muestra original permanece intacta.

In [3]:
def q02():
    sin_duplicados = df.unique()  # versión lógica, NO sobrescribe el CSV.
    despues = sin_duplicados.height
    return [{"antes": TOTAL, "despues": despues,
             "eliminados": TOTAL - despues,
             "reduccion_pct": 100 * (TOTAL - despues) / TOTAL}]


r02 = medir("Q2", q02, TIEMPOS, MOTOR)



Q2: 1 filas → q02_duplicados.csv


  {'antes': 600000, 'despues': 468970, 'eliminados': 131030, 'reduccion_pct': 21.838333333333335}


Q2: 0.387 s (incluye cómputo/materialización, no exportación)



## Q3 — Transforma Label en una de 8 familias, comprueba la cobertura de las 34 etiquetas.

In [4]:
def q03():
    global clasificado
    clasificado = familia(df)
    t = clasificado.group_by("Attack_Family").agg(
        pl.len().alias("cantidad"), pl.col("Label").n_unique().alias("clases"))
    assert clasificado.select(pl.col("Label").n_unique()).item() == 34
    assert t["clases"].sum() == 34 and t["cantidad"].sum() == TOTAL
    return filas_tabla(t.sort("Attack_Family"))


r03 = medir("Q3", q03, TIEMPOS, MOTOR)



Q3: 8 filas → q03_familias.csv


  {'Attack_Family': 'Benign', 'cantidad': 14087, 'clases': 1}


  {'Attack_Family': 'BruteForce', 'cantidad': 169, 'clases': 1}


  {'Attack_Family': 'DDoS', 'cantidad': 435902, 'clases': 12}


  {'Attack_Family': 'DoS', 'cantidad': 100627, 'clases': 4}


  {'Attack_Family': 'Mirai', 'cantidad': 33788, 'clases': 3}


  {'Attack_Family': 'Recon', 'cantidad': 8862, 'clases': 5}


  {'Attack_Family': 'Spoofing', 'cantidad': 6242, 'clases': 2}


  {'Attack_Family': 'Web-based', 'cantidad': 323, 'clases': 6}


Q3: 0.094 s (incluye cómputo/materialización, no exportación)



## Q4 — Filtra Label != Benign sin reemplazar el dataset original.

In [5]:
def q04():
    maliciosos = df.filter(pl.col("Label") != "Benign")
    return [{"cantidad": maliciosos.height, "porcentaje": 100 * maliciosos.height / TOTAL}]


r04 = medir("Q4", q04, TIEMPOS, MOTOR)



Q4: 1 filas → q04_malicioso.csv


  {'cantidad': 585913, 'porcentaje': 97.65216666666667}


Q4: 0.017 s (incluye cómputo/materialización, no exportación)



## Q5 — Crea Traffic_Type y resume benigno vs malicioso.

In [6]:
def q05():
    global tipado
    tipado = tipo(df)
    t = tipado.group_by("Traffic_Type").agg(pl.len().alias("cantidad"))
    t = t.with_columns((pl.col("cantidad") * 100 / TOTAL).alias("porcentaje"))
    return filas_tabla(t.sort("Traffic_Type"))


r05 = medir("Q5", q05, TIEMPOS, MOTOR)



Q5: 2 filas → q05_tipo_trafico.csv


  {'Traffic_Type': 'Benign', 'cantidad': 14087, 'porcentaje': 2.3478333333333334}


  {'Traffic_Type': 'Malicious', 'cantidad': 585913, 'porcentaje': 97.65216666666667}


Q5: 0.025 s (incluye cómputo/materialización, no exportación)



## Q6 — Agrupa por familia y ordena por cantidad descendente.

In [7]:
def q06():
    t = clasificado.group_by("Attack_Family").agg(pl.len().alias("cantidad"))
    t = t.with_columns((pl.col("cantidad") * 100 / TOTAL).alias("porcentaje"))
    return filas_tabla(t.sort(["cantidad", "Attack_Family"], descending=[True, False]))


r06 = medir("Q6", q06, TIEMPOS, MOTOR)



Q6: 8 filas → q06_distribucion_familias.csv


  {'Attack_Family': 'DDoS', 'cantidad': 435902, 'porcentaje': 72.65033333333334}


  {'Attack_Family': 'DoS', 'cantidad': 100627, 'porcentaje': 16.771166666666666}


  {'Attack_Family': 'Mirai', 'cantidad': 33788, 'porcentaje': 5.631333333333333}


  {'Attack_Family': 'Benign', 'cantidad': 14087, 'porcentaje': 2.3478333333333334}


  {'Attack_Family': 'Recon', 'cantidad': 8862, 'porcentaje': 1.477}


  {'Attack_Family': 'Spoofing', 'cantidad': 6242, 'porcentaje': 1.0403333333333333}


  {'Attack_Family': 'Web-based', 'cantidad': 323, 'porcentaje': 0.05383333333333333}


  {'Attack_Family': 'BruteForce', 'cantidad': 169, 'porcentaje': 0.028166666666666666}


Q6: 0.013 s (incluye cómputo/materialización, no exportación)



## Q7 — Top 10 de clases maliciosas; porcentaje sobre los 600.000 registros totales.

In [8]:
def q07():
    t = df.filter(pl.col("Label") != "Benign").group_by("Label").agg(
        pl.len().alias("cantidad"))
    t = t.with_columns((pl.col("cantidad") * 100 / TOTAL).alias("porcentaje"))
    return filas_tabla(t.sort(["cantidad", "Label"], descending=[True, False]).head(10))


r07 = medir("Q7", q07, TIEMPOS, MOTOR)



Q7: 10 filas → q07_top10_ataques.csv


  {'Label': 'DDoS-ICMP_Flood', 'cantidad': 92356, 'porcentaje': 15.392666666666667}


  {'Label': 'DDoS-UDP_Flood', 'cantidad': 69419, 'porcentaje': 11.569833333333333}


  {'Label': 'DDoS-TCP_Flood', 'cantidad': 57689, 'porcentaje': 9.614833333333333}


  {'Label': 'DDoS-PSHACK_FLOOD', 'cantidad': 52521, 'porcentaje': 8.7535}


...

  {'Label': 'DDoS-SynonymousIP_Flood', 'cantidad': 46151, 'porcentaje': 7.691833333333333}


  {'Label': 'DoS-UDP_Flood', 'cantidad': 39416, 'porcentaje': 6.569333333333334}


  {'Label': 'DoS-TCP_Flood', 'cantidad': 34265, 'porcentaje': 5.710833333333333}


  {'Label': 'DoS-SYN_Flood', 'cantidad': 26023, 'porcentaje': 4.337166666666667}


Q7: 0.050 s (incluye cómputo/materialización, no exportación)



## Q8 — Sustituye los 13 Rate infinitos por nulos SOLO en Rate; IAT usa las 600.000 filas.

In [9]:
def q08():
    # Rate=+inf -> NULL solo para esa columna: IAT incluye las 600k filas.
    t = limpio_rate(clasificado).group_by("Attack_Family").agg(
        pl.len().alias("registros"), pl.col("Rate").null_count().alias("rate_inf_excluidos"),
        pl.col("Rate").mean().alias("rate_media"),
        pl.col("Rate").median().alias("rate_mediana"),
        pl.col("Rate").min().alias("rate_min"), pl.col("Rate").max().alias("rate_max"),
        pl.col("IAT").mean().alias("iat_media"),
        pl.col("IAT").median().alias("iat_mediana"),
        pl.col("IAT").min().alias("iat_min"), pl.col("IAT").max().alias("iat_max"))
    assert t["rate_inf_excluidos"].sum() == 13
    return filas_tabla(t.sort("Attack_Family"))


r08 = medir("Q8", q08, TIEMPOS, MOTOR)



Q8: 8 filas → q08_rate_iat_familia.csv


  {'Attack_Family': 'Benign', 'registros': 14087, 'rate_inf_excluidos': 2, 'rate_media': 2667.4811567854676, 'rate_mediana': 171.85122057148476, 'rate_min': 11.18366844132986, 'rate_max': 723155.8620689656, 'iat_media': 0.009185431232298744, 'iat_mediana': 0.0066066026687622, 'iat_min': 1.5020370483398435e-06, 'iat_max': 0.0896720886230468}


  {'Attack_Family': 'BruteForce', 'registros': 169, 'rate_inf_excluidos': 0, 'rate_media': 6106.94353291149, 'rate_mediana': 118.88785271900112, 'rate_min': 9.313311830335037, 'rate_max': 582542.2222222222, 'iat_media': 0.01518694296391048, 'iat_mediana': 0.0094049930572509, 'iat_min': 1.811981201171875e-06, 'iat_max': 0.1103171110153198}


  {'Attack_Family': 'DDoS', 'registros': 435902, 'rate_inf_excluidos': 3, 'rate_media': 32856.65569552071, 'rate_mediana': 29155.456694008062, 'rate_min': 0.0616814565794651, 'rate_max': 7340032.0, 'iat_media': 0.00015632125833893305, 'iat_mediana': 3.471851348876953e-05, 'iat_min': 0.0, 'iat_max': 16.21232791900635}


  {'Attack_Family': 'DoS', 'registros': 100627, 'rate_inf_excluidos': 4, 'rate_media': 22403.689201332756, 'rate_mediana': 21829.41605079629, 'rate_min': 0.000428338067839, 'rate_max': 729444.1739130435, 'iat_media': 0.023857303023602387, 'iat_mediana': 4.632949829101562e-05, 'iat_min': 0.0, 'iat_max': 2334.6046034812925}


  {'Attack_Family': 'Mirai', 'registros': 33788, 'rate_inf_excluidos': 4, 'rate_media': 5615.774688897283, 'rate_mediana': 4628.018781232153, 'rate_min': 0.5485152615709177, 'rate_max': 1677721.6, 'iat_media': 0.00046951247128418214, 'iat_mediana': 0.00021839976310725, 'iat_min': 1.0728836059570312e-06, 'iat_max': 1.8231033301353452}


  {'Attack_Family': 'Recon', 'registros': 8862, 'rate_inf_excluidos': 0, 'rate_media': 11578.112570365392, 'rate_mediana': 107.69648444296658, 'rate_min': 0.0833796569774162, 'rate_max': 2621440.0, 'iat_media': 0.015865826020254728, 'iat_mediana': 0.01047899723052975, 'iat_min': 5.006790161132813e-07, 'iat_max': 11.998069310188294}


  {'Attack_Family': 'Spoofing', 'registros': 6242, 'rate_inf_excluidos': 0, 'rate_media': 31845.4150744199, 'rate_mediana': 639.8389185036615, 'rate_min': 0.0333738298136531, 'rate_max': 1997287.619047619, 'iat_media': 0.011380708923152969, 'iat_mediana': 0.00181715488433835, 'iat_min': 5.006790161132813e-07, 'iat_max': 29.990078592300414}


  {'Attack_Family': 'Web-based', 'registros': 323, 'rate_inf_excluidos': 0, 'rate_media': 6495.370269983226, 'rate_mediana': 79.68629358300149, 'rate_min': 10.03336077914858, 'rate_max': 371177.34513274336, 'iat_media': 0.019921347452760065, 'iat_mediana': 0.014647102355957, 'iat_min': 2.884864807128906e-06, 'iat_max': 0.1025820970535278}


Q8: 0.058 s (incluye cómputo/materialización, no exportación)



## Q9 — Mapea el número IP Protocol Type; no usa las columnas agregadas TCP/UDP/ICMP.

In [10]:
def q09():
    t = clasificado.with_columns(pl.col("Protocol Type").replace_strict(PROTOCOLOS)
                                  .alias("protocolo"))
    t = t.group_by("Attack_Family", "Protocol Type", "protocolo").agg(
        pl.len().alias("cantidad"))
    totales = clasificado.group_by("Attack_Family").agg(pl.len().alias("total_familia"))
    t = t.join(totales, on="Attack_Family").with_columns(
        (100 * pl.col("cantidad") / pl.col("total_familia")).alias("porcentaje_familia"))
    return filas_tabla(t.sort(["Attack_Family", "Protocol Type"]))


r09 = medir("Q9", q09, TIEMPOS, MOTOR)



Q9: 29 filas → q09_protocolos_familia.csv


  {'Attack_Family': 'Benign', 'Protocol Type': 0, 'protocolo': 'HOPOPT', 'cantidad': 31, 'total_familia': 14087, 'porcentaje_familia': 0.22006104919429262}


  {'Attack_Family': 'Benign', 'Protocol Type': 1, 'protocolo': 'ICMP', 'cantidad': 3, 'total_familia': 14087, 'porcentaje_familia': 0.021296230567189607}


  {'Attack_Family': 'Benign', 'Protocol Type': 6, 'protocolo': 'TCP', 'cantidad': 12887, 'total_familia': 14087, 'porcentaje_familia': 91.48150777312415}


  {'Attack_Family': 'Benign', 'Protocol Type': 17, 'protocolo': 'UDP', 'cantidad': 1166, 'total_familia': 14087, 'porcentaje_familia': 8.27713494711436}


...

  {'Attack_Family': 'DDoS', 'Protocol Type': 17, 'protocolo': 'UDP', 'cantidad': 73144, 'total_familia': 435902, 'porcentaje_familia': 16.77991842203064}


  {'Attack_Family': 'DoS', 'Protocol Type': 1, 'protocolo': 'ICMP', 'cantidad': 116, 'total_familia': 100627, 'porcentaje_familia': 0.11527721188150297}


  {'Attack_Family': 'DoS', 'Protocol Type': 6, 'protocolo': 'TCP', 'cantidad': 61228, 'total_familia': 100627, 'porcentaje_familia': 60.846492492074695}


  ... 17 filas adicionales en CSV


Q9: 0.085 s (incluye cómputo/materialización, no exportación)



## Q10 — Compara los dos tipos de tráfico. Rate excluye 13 infinitos; las demás métricas no.

In [11]:
def q10():
    t = limpio_rate(tipado).group_by("Traffic_Type").agg(
        pl.len().alias("registros"), pl.col("Rate").null_count().alias("rate_inf_excluidos"),
        pl.col("Rate").mean().alias("rate_media"),
        pl.col("Rate").median().alias("rate_mediana"),
        pl.col("IAT").mean().alias("iat_media"),
        pl.col("IAT").median().alias("iat_mediana"),
        pl.col("ack_flag_number").mean().alias("ack_flag_media"),
        pl.col("syn_flag_number").mean().alias("syn_flag_media"),
        pl.col("Tot sum").mean().alias("tot_sum_media"),
        pl.col("AVG").mean().alias("avg_media"))
    assert t["rate_inf_excluidos"].sum() == 13
    return filas_tabla(t.sort("Traffic_Type"))


r10 = medir("Q10", q10, TIEMPOS, MOTOR)



Q10: 2 filas → q10_perfil_trafico.csv


  {'Traffic_Type': 'Benign', 'registros': 14087, 'rate_inf_excluidos': 2, 'rate_media': 2667.4811567854676, 'rate_mediana': 171.85122057148476, 'iat_media': 0.009185431232298744, 'iat_mediana': 0.0066066026687622, 'ack_flag_media': 0.8030792771901596, 'syn_flag_media': 0.013679278767658126, 'tot_sum_media': 6088.058777596366, 'avg_media': 609.1035888092252}


  {'Traffic_Type': 'Malicious', 'registros': 585913, 'rate_inf_excluidos': 11, 'rate_media': 29135.84098392661, 'rate_mediana': 25247.11972551616, 'iat_media': 0.004617299823977632, 'iat_mediana': 4.008054733276368e-05, 'ack_flag_media': 0.11434509125214834, 'syn_flag_media': 0.2121034588419937, 'tot_sum_media': 11095.438104291934, 'avg_media': 120.07703692400342}


Q10: 0.070 s (incluye cómputo/materialización, no exportación)



## Tiempos por consulta

Tiempos de pared (segundos). Comparar solo con atención a carga, caché, hilos y materialización.

In [12]:
cerrar(MOTOR, TIEMPOS)


Tiempos medidos: [{'Consulta': 'Q1', 'Tiempo': 0.43}, {'Consulta': 'Q2', 'Tiempo': 0.387}, {'Consulta': 'Q3', 'Tiempo': 0.094}, {'Consulta': 'Q4', 'Tiempo': 0.017}, {'Consulta': 'Q5', 'Tiempo': 0.025}, {'Consulta': 'Q6', 'Tiempo': 0.013}, {'Consulta': 'Q7', 'Tiempo': 0.05}, {'Consulta': 'Q8', 'Tiempo': 0.058}, {'Consulta': 'Q9', 'Tiempo': 0.085}, {'Consulta': 'Q10', 'Tiempo': 0.07}]
